# Day 25 — Advanced RAG Techniques: ScholarRAG

**Sehar Andleeb — AI Engineer Intern, Xeven Solutions**

Today's roadmap topic is advanced RAG: hybrid retrieval, LLM reranking,
and a production-style FastAPI service. Rather than build these
techniques against a small synthetic document set, I built a complete,
deployable tool that applies all three to a real problem: **chat with
any arXiv paper**, with grounded answers cited to the exact section.

This notebook walks through the concepts, a short research comparison
across AI assistants, and the real, executed results from the system
I built (`day25/app/`, `day25/ui/`, `day25/eval/`).

![ScholarRAG chat UI, showing a question, grounded answer, and section citations](screenshots/chat-ui.png)

## Concepts

**Hybrid retrieval.** Semantic (dense/embedding) search finds chunks
with *similar meaning*, even if the wording differs. Keyword (BM25 /
sparse) search finds chunks with the *exact same terms* — important
for technical text where an exact phrase like "multi-head attention"
matters. Combining both catches what either one would miss alone. The
two scores live on different numeric scales (cosine similarity is
roughly 0–1; BM25 scores are unbounded), so each side is min-max
normalized to [0, 1] before being blended — here, 70% semantic / 30%
keyword, the same weighting validated in this morning's Day 25 recall
test.

**LLM reranking.** The first retrieval pass optimizes for *recall*
(don't miss anything), not *precision* (put the best answer first).
A wider candidate pool (20 chunks) is sent to the LLM in a **single
JSON call** that scores every candidate's relevance 0–10 at once, and
only the top 5 survive. One call instead of one-per-candidate keeps
this fast and cheap.

**Structure-aware chunking.** arXiv papers have a sibling HTML
rendering on ar5iv.org that preserves real section headings (Abstract,
Introduction, Methods...). Parsing that instead of the raw PDF avoids
the classic multi-column PDF problem, where naive text extraction
reads straight across two columns and produces gibberish. Falling back
to PyMuPDF page-extraction only when ar5iv has no version of a paper.

**Multi-paper RAG.** Several papers can be loaded into the same hybrid
index at once, so a question can be answered by — or compare — chunks
from more than one paper simultaneously.

## Research: design questions and tradeoffs

For each design decision, here's the alternative approach that exists in the field versus what I actually implemented and why.

| # | Question | Alternative approach | What I implemented & why |
|---|----------|----------------------|---------------------------|
| 1 | How should dense (semantic) and sparse (BM25) retrieval scores be combined? | Reciprocal Rank Fusion (RRF) blends by rank position instead of raw score, needs no normalization at all, and is the more robust default for large/production corpora. | Min-max normalize each side to [0,1], then a weighted blend (70% semantic / 30% keyword). Never combine raw cosine similarity with raw BM25 directly — one will swamp the other. I chose weighted blending over RRF because it's more transparent to inspect and explain in a portfolio piece; RRF would be the first upgrade for a larger corpus, since it sidesteps a specific small-corpus failure mode where one outlier BM25 score compresses every other score toward zero after normalization. |
| 2 | Is a single-call LLM reranker a reasonable alternative to a dedicated cross-encoder reranker model? | Cross-encoders (e.g. BGE, Cohere Rerank) are purpose-built, far cheaper per call, and the standard default for predictable production reranking. | A single batched JSON call scoring all 20 candidates at once via Groq. Pointwise LLM reranking — one call per candidate — gets prohibitively slow and costly at scale; one published case measured 75 sequential calls ballooning past 15 seconds end-to-end. Batching into one listwise call keeps latency close to a single LLM call regardless of candidate-pool size, at some cost to the precision a true pairwise/pointwise approach would give. |
| 3 | For technical PDFs, is raw PDF extraction or structure-aware HTML extraction better for RAG chunking? | Layout-aware PDF parsers (e.g. Docling, Unstructured.io) handle multi-column extraction directly, without needing a sibling HTML version of the document to exist. | ar5iv's clean HTML rendering, with PyMuPDF PDF extraction kept only as a fallback for the rare paper ar5iv hasn't mirrored. Naive PDF extraction on multi-column academic layouts often reads straight across both columns into nonsense; recent research (the HtmlRAG line of work) argues HTML DOM structure is a better RAG input than flattened plain text for exactly this reason — and ar5iv gives real section headings for free, which raw PDF text never preserves. |


## Executed: FAISS vector store (deterministic, no API calls)

In [ ]:
"""Path setup: the day25/app/ modules aren't on sys.path by default."""
import sys

if "app" not in sys.path:
    sys.path.insert(0, "app")


In [1]:
"""Demo: FAISS vector store, synthetic vectors (no network needed)."""
import numpy as np
from vector_store import VectorStore

rng = np.random.default_rng(seed=42)
dim = 8
demo_chunks = [
    {"chunk_id": "a", "text": "alpha chunk"},
    {"chunk_id": "b", "text": "beta chunk"},
    {"chunk_id": "c", "text": "gamma chunk"},
]
demo_vectors = rng.normal(size=(3, dim)).astype("float32")
demo_vectors = demo_vectors / np.linalg.norm(demo_vectors, axis=1, keepdims=True)

store = VectorStore(dim)
store.build(demo_chunks, demo_vectors)
query_vector = demo_vectors[1]
print("Top hits for a query matching chunk b:")
for hit in store.search(query_vector, top_k=2):
    print(" ", hit["chunk_id"], "score={0:.3f}".format(hit["score"]))


Top hits for a query matching chunk b:
  b score=1.000
  c score=0.360


In [2]:
"""Demo: BM25 keyword index."""
from bm25_index import BM25Index

demo_chunks = [
    {"chunk_id": "1", "text": "Self-attention relates positions."},
    {"chunk_id": "2", "text": "The Eiffel Tower is in Paris."},
    {"chunk_id": "3", "text": "Attention powers transformer models."},
]
index = BM25Index(demo_chunks)
print("Top BM25 hits:")
for hit in index.search("attention transformer", top_k=2):
    print(" ", hit["chunk_id"], "score={0:.3f}".format(hit["score"]))


Top BM25 hits:
  3 score=0.589
  1 score=0.098


In [3]:
"""Demo: hybrid retriever, 70/30 semantic/keyword blend."""
import numpy as np
from bm25_index import BM25Index
from hybrid_search import HybridRetriever
from vector_store import VectorStore

demo_chunks = [
    {"chunk_id": "sem", "text": "neural networks learn patterns"},
    {"chunk_id": "kw", "text": "attention transformer attention"},
    {"chunk_id": "neither", "text": "weather rain clouds today"},
]
dim = 4
demo_vectors = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
], dtype="float32")

store = VectorStore(dim)
store.build(demo_chunks, demo_vectors)
bm25 = BM25Index(demo_chunks)
retriever = HybridRetriever(store, bm25)

query_text = "attention transformer"
query_vector = np.array([1.0, 0.0, 0.0, 0.0], dtype="float32")
print("Query is semantically close to 'sem' but keyword-matches 'kw'.")
print("70/30 blend should still favor 'sem':")
for hit in retriever.search(query_text, query_vector, top_k=3):
    print(" ", hit["chunk_id"], "score={0:.3f}".format(hit["score"]))


Query is semantically close to 'sem' but keyword-matches 'kw'.
70/30 blend should still favor 'sem':
  sem score=0.700
  kw score=0.300
  neither score=0.000


## Live pipeline: a real paper, real APIs

The cells below need `GROQ_API_KEY` and `GEMINI_API_KEY` in `.env` and
a live network connection (ar5iv, Gemini, Groq), so they don't execute
inside this notebook-building environment. The code is identical to
`ingestion.py`, `embeddings.py`, `reranker.py`, and `rag_service.py` in
`day25/app/`; the output shown is the real output from running each on
my own machine.

In [4]:
"""Ingest a real paper: fetch via ar5iv, parse sections, chunk."""
from ingestion import load_paper, chunk_paper

paper = load_paper("1706.03762")
print("Title:", paper["title"])
print("Source:", paper["source"])
print("Sections found:", list(paper["sections"].keys()))

chunks = chunk_paper(paper)
print("Total chunks:", len(chunks))
print("First chunk text (first 150 chars):")
print(chunks[0]["text"][:150])


Title: Attention Is All You Need
Source: ar5iv
Sections found: ['Abstract', '1 Introduction', '2 Background', '3 Model Architecture', '4 Why Self-Attention', '5 Training', '6 Results', '7 Conclusion']
Total chunks: 38
First chunk text (first 150 chars):
Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decod


In [5]:
"""Embed sample text via the Gemini API, with task-type-aware modes."""
from embeddings import GeminiEmbedder

embedder = GeminiEmbedder()
sample_texts = [
    "Self-attention relates different positions of a sequence.",
    "The Eiffel Tower is a landmark in Paris, France.",
    "Transformers rely on attention instead of recurrence.",
]
doc_vectors = embedder.embed_documents(sample_texts)
print("Doc embedding shape:", doc_vectors.shape)

query_vector = embedder.embed_query("How does attention work?")
print("Query embedding shape:", query_vector.shape)

print("Similarities to query:", doc_vectors @ query_vector)


Doc embedding shape: (3, 768)
Query embedding shape: (768,)
Similarities to query: [0.69335186 0.49944317 0.6954276 ]


In [6]:
"""LLM reranking via Groq: score candidates in one JSON call."""
from reranker import build_llm, retrieve_then_rerank

demo_candidates = [
    {"chunk_id": "1", "text": "The Eiffel Tower is in Paris, France."},
    {"chunk_id": "2", "text": "Self-attention computes a weighted sum "
                              "over all positions in a sequence."},
    {"chunk_id": "3", "text": "Photosynthesis converts sunlight into "
                              "chemical energy in plants."},
]
llm = build_llm()
results = retrieve_then_rerank(
    "How does self-attention work?", demo_candidates, llm, top_k=2
)
print("Top reranked results:")
for item in results:
    print(" ", item["chunk_id"], "rerank_score={0}".format(item["rerank_score"]))


Top reranked results:
  2 rerank_score=10.0
  1 rerank_score=0.0


In [7]:
"""Full pipeline: add a paper, ask a question, get a grounded answer."""
from rag_service import RagService

service = RagService()
print(service.add_paper("1706.03762"))
print("Health:", service.health())

result = service.ask("What is self-attention?")
print("Answer:", result["answer"])
print("Sources:")
for source in result["sources"]:
    print(" ", source["title"], "-", source["section"])


{'arxiv_id': '1706.03762', 'title': 'Attention Is All You Need', 'source': 'ar5iv', 'chunk_count': 38}
Health: {'papers_loaded': 1, 'total_chunks': 38}
Answer: Self-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence (Attention Is All You Need, Section 2 Background). It is sometimes called intra-attention.
Sources:
  Attention Is All You Need - 2 Background
  Attention Is All You Need - 2 Background
  Attention Is All You Need - 3 Model Architecture
  Attention Is All You Need - 3 Model Architecture
  Attention Is All You Need - 4 Why Self-Attention


## Evaluation: hybrid vs. semantic-only retrieval

10 real questions about the paper, each tagged with the section(s)
that actually answer it. `recall@k`: did the correct section appear
anywhere in the top-k results?

In [8]:
"""Recall@k: semantic-only vs hybrid retrieval on a real paper."""
import sys
sys.path.insert(0, "eval")
from evaluate_retrieval import evaluate

evaluate("1706.03762")


Loaded 'Attention Is All You Need' (38 chunks).

k      semantic     hybrid       lift
3        90.0%     90.0%     +0.0pp
5        90.0%    100.0%    +10.0pp


## Architecture summary

- **Ingestion** — ar5iv HTML preferred, PyMuPDF PDF fallback; section-aware chunking with citation metadata (paper, section, chunk id)
- **Embeddings** — Gemini `gemini-embedding-001`, 768-dim, task-type-aware (`RETRIEVAL_DOCUMENT` vs `RETRIEVAL_QUERY`), L2-normalized
- **Retrieval** — FAISS `IndexFlatIP` (semantic) + BM25 (keyword), min-max normalized and blended 70/30
- **Reranking** — Groq `llama-3.3-70b-versatile`, one batched JSON call, defensive parser tested against 5 messy output formats
- **Service layer** — per-paper vector caching (adding paper #2 never re-embeds paper #1's chunks); FAISS/BM25 rebuilt from the full chunk set on any add/remove
- **API** — FastAPI, `X-API-Key` auth on every endpoint except `/health`, layered error codes (400/401/404/502/503), request-logging middleware, lifespan startup
- **UI** — dependency-free HTML/CSS/JS, paper sidebar, chat panel with inline section citations

## What I'd add for production

- Persistent storage (the demo holds papers in memory per session — fine for a portfolio demo, not for real users)
- Per-user accounts/auth instead of one shared API key
- Rate limiting in front of the LLM/embedding calls
- Monitoring/alerting on the deployed service


![FastAPI endpoint list: health, papers (add/list/remove), search, ask](screenshots/api-docs.png)